# Phase 3 - Feature Engineering
This notebook creates model-ready features from `data/processed/cleaned.csv`, encodes categorical variables, and saves outputs to:
- `data/processed/features.csv`
- `models/encoders.pkl`

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
INPUT_CSV = ROOT / "data" / "processed" / "cleaned.csv"
OUTPUT_CSV = ROOT / "data" / "processed" / "features.csv"
ENCODERS_PATH = ROOT / "models" / "encoders.pkl"

df = pd.read_csv(INPUT_CSV)
print(f"Loaded: {INPUT_CSV}")
print(f"Shape: {df.shape}")

Loaded: C:\Users\VASU MONPARA\OneDrive\Desktop\GujEstateAI Gujarat Real Estate Prediction System\data\processed\cleaned.csv
Shape: (13425, 42)


In [2]:
def safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    result = numerator.astype(float) / denominator.replace({0: np.nan}).astype(float)
    return result.replace([np.inf, -np.inf], np.nan)

# === Target for Module 1 ===
# Duration in months (regression target)
df['duration_months'] = (df['EndProjectYear'] - df['startProjectYear']) * 12 + (df['EndProjectMonth'] - df['startProjectMonth'])

# === Target for Module 2 ===
# Already exists: totalEstimatedCost

# === Derived Features ===
df['booking_rate'] = safe_divide(df['bookedUnits'], df['totalUnits'])
df['cost_per_unit'] = safe_divide(df['totalEstimatedCost'], df['totalUnits'])
df['land_cost_ratio'] = safe_divide(df['totalLandCost'], df['totalEstimatedCost'])
df['sell_dev_ratio'] = safe_divide(df['totalSellingAmount'], df['totalDevelopCost'])
df['is_redevelop'] = (df['underRedevelopment'].astype(str).str.strip().str.upper() == 'YES').astype(int)
df['log_cost'] = np.log1p(df['totalEstimatedCost'].clip(lower=0))
df['log_units'] = np.log1p(df['totalUnits'].clip(lower=0))
df['start_quarter'] = df['start_month'].apply(lambda m: (int(m) - 1) // 3 + 1 if pd.notna(m) else np.nan)

df[['duration_months', 'booking_rate', 'cost_per_unit', 'land_cost_ratio', 'sell_dev_ratio', 'is_redevelop', 'log_cost', 'log_units', 'start_quarter']].head()

,duration_months,booking_rate,cost_per_unit,land_cost_ratio,sell_dev_ratio,is_redevelop,log_cost,log_units,start_quarter
0,51.0,0.957755,591931.508391,0.001129,2.197096,0,21.439013,8.148156,2
1,78.0,0.214375,765131.842500,0.154214,1.438876,0,21.618710,8.071219,1
2,50.0,0.000000,935156.250000,0.000000,0.541353,0,21.596231,7.848153,4
3,69.0,0.000000,559499.067487,0.000000,0.226409,0,20.927823,7.693482,2
4,66.0,0.085256,727850.373621,0.042260,1.448031,0,21.095749,7.598399,3


In [3]:
encoders = {}

for col in ['projectType', 'distName', 'promoterType']:
    le = LabelEncoder()
    df[col + '_enc'] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

ENCODERS_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(encoders, ENCODERS_PATH)
print(f"Saved encoders: {ENCODERS_PATH}")

Saved encoders: C:\Users\VASU MONPARA\OneDrive\Desktop\GujEstateAI Gujarat Real Estate Prediction System\models\encoders.pkl


In [4]:
# Module 1 - Duration Prediction features
FEATURES_DURATION = [
    'projectType_enc', 'distName_enc', 'promoterType_enc',
    'totalUnits', 'log_cost', 'totalLandCost',
    'is_redevelop', 'start_year', 'start_quarter',
    'land_cost_ratio', 'avgCostPerUnit'
]

# Module 2 - Cost Prediction features
FEATURES_COST = [
    'projectType_enc', 'distName_enc', 'promoterType_enc',
    'totalUnits', 'duration_months', 'totalLandCost',
    'totalCarpetArea_form3A', 'avgCostPerSqFt',
    'is_redevelop', 'start_year'
]

# Module 4 - Clustering features
FEATURES_CLUSTER = [
    'log_cost', 'log_units', 'duration_months',
    'avgCostPerSqFt', 'booking_rate',
    'land_cost_ratio', 'projectType_enc', 'distName_enc'
]

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved features: {OUTPUT_CSV}")
print(f"Final shape: {df.shape}")

c:\Users\VASU MONPARA\OneDrive\Desktop\GujEstateAI Gujarat Real Estate Prediction System\gpu_env\Lib\site-packages\pandas\core\internals\blocks.py:2540: RuntimeWarning: invalid value encountered in cast
  values = values.astype(str)


Saved features: C:\Users\VASU MONPARA\OneDrive\Desktop\GujEstateAI Gujarat Real Estate Prediction System\data\processed\features.csv
Final shape: (13425, 52)


In [5]:
required_columns = set(FEATURES_DURATION + FEATURES_COST + FEATURES_CLUSTER + ['duration_months', 'totalEstimatedCost'])
missing = sorted([c for c in required_columns if c not in df.columns])
if missing:
    print('Missing columns:', missing)
else:
    print('All required feature columns are present.')

All required feature columns are present.
